# NY Bus Delay Prediction — ML Tutorial

This notebook trains a **RF classifier** to predict whether a NYC bus will be delayed,
using the [Bus Breakdown and Delays dataset from OpenML](https://www.openml.org/search?type=data&status=active&id=43484).

## Pipeline
1. Load dataset from OpenML
2. Exploratory Data Analysis
3. Feature Engineering
4. Train / Evaluate Random Forest
5. Save model for deployment

In [ ]:
import openml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay
)

## 1. Load Dataset from OpenML

In [ ]:
# Load dataset (ID 43484 = Bus Breakdown and Delays NYC)
dataset = openml.datasets.get_dataset(43484)
X, y, categorical_indicator, attribute_names = dataset.get_data(
    dataset_format='dataframe',
    target=dataset.default_target_attribute
)

# Combine into one df for EDA
df = X.copy()

# Ensure we have a usable target column
fallback_target_col = 'Breakdown_or_Running_Late'
if y is not None:
    target_series = y.copy() if hasattr(y, 'copy') else pd.Series(y)
else:
    target_series = None

if target_series is None or (hasattr(target_series, 'isna') and target_series.isna().all()):
    if fallback_target_col in df.columns:
        print(
            "Default OpenML target missing; falling back to '",
            f"{fallback_target_col}'.",
            sep=''
        )
        target_series = df[fallback_target_col].copy()
    else:
        raise ValueError('No usable target column found in dataset.')

if target_series.isna().all():
    raise ValueError('Fallback target column is empty; please inspect the dataset.')

df['target'] = target_series
# Keep y aligned with whatever target we finally selected for downstream steps
y = target_series

print(f'Dataset: {dataset.name}')
print(f'Shape: {df.shape}')
print(f"Target column used: {'Breakdown_or_Running_Late' if target_series.name == fallback_target_col else dataset.default_target_attribute}")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\nTotal rows: {len(df)}')

In [ ]:
print('=== Target Distribution ===')
target_counts = df['target'].dropna().value_counts()
print(target_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Target distribution
if len(target_counts) > 0:
    colors = sns.color_palette('Set2', n_colors=len(target_counts))
    target_counts.plot(kind='bar', ax=axes[0], color=colors)
    axes[0].set_xlabel('Target Class')
else:
    axes[0].text(0.5, 0.5, 'No target labels available', ha='center', va='center', fontsize=12)
    axes[0].set_xlabel('')
    axes[0].set_xticks([])

axes[0].set_title('Target Distribution (Delayed vs Not Delayed)', fontsize=13)
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Missing values
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]
if len(missing_pct) > 0:
    missing_pct.plot(kind='bar', ax=axes[1], color='#f59e0b')
    axes[1].set_title('Missing Values (%)', fontsize=13)
    axes[1].set_ylabel('% Missing')
    axes[1].tick_params(axis='x', rotation=45)
else:
    axes[1].text(0.5, 0.5, 'No missing values!', ha='center', va='center', fontsize=14)
    axes[1].set_title('Missing Values')

plt.tight_layout()
# plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Feature Engineering

In [ ]:
df_model = X.copy()

# --- Encode categorical columns ---
label_encoders = {}
categorical_cols = df_model.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns ({len(categorical_cols)}): {categorical_cols}')

for col in categorical_cols:
    le = LabelEncoder()
    # Fill NaN before encoding
    df_model[col] = df_model[col].fillna('Unknown')
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = le

# --- Fill remaining NaN ---
df_model = df_model.fillna(df_model.median(numeric_only=True))

# --- Target ---
if y.dtype == 'object' or str(y.dtype) == 'category':
    le_target = LabelEncoder()
    y_encoded = le_target.fit_transform(y.astype(str))
    target_classes = le_target.classes_.tolist()
else:
    y_encoded = y.values
    target_classes = list(map(str, np.unique(y_encoded)))

print(f'\nTarget classes: {target_classes}')
print(f'Feature matrix shape: {df_model.shape}')

In [ ]:
# Select features to use (drop very high cardinality or id-like columns if any)
feature_cols = df_model.columns.tolist()
print(f'Features used: {feature_cols}')

## 4. Train / Evaluate Random Forest

In [ ]:
X_values = df_model[feature_cols].to_numpy(copy=True)
y_values = np.asarray(y_encoded)

# Guarantee numeric targets for downstream steps
if not np.issubdtype(y_values.dtype, np.number):
    y_values_str = y_values.astype(str)
    if 'target_classes' in globals():
        class_to_idx = {str(cls): idx for idx, cls in enumerate(target_classes)}
        unseen = [cls for cls in np.unique(y_values_str) if cls not in class_to_idx]
        if unseen:
            target_classes = target_classes + unseen
            class_to_idx = {str(cls): idx for idx, cls in enumerate(target_classes)}
        y_values = np.array([class_to_idx[label] for label in y_values_str], dtype=int)
    else:
        le_tmp = LabelEncoder()
        y_values = le_tmp.fit_transform(y_values_str)
        target_classes = le_tmp.classes_.tolist()
else:
    y_values = y_values.astype(int)

X_train_arr, X_test_arr, y_train, y_test = train_test_split(
    X_values,
    y_values,
    test_size=0.2,
    random_state=42,
    stratify=y_values
)

# Wrap arrays back into DataFrames for downstream convenience
X_train = pd.DataFrame(X_train_arr, columns=feature_cols)
X_test = pd.DataFrame(X_test_arr, columns=feature_cols)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train class balance: {np.bincount(y_train)}')

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
print('Random Forest trained!')

In [ ]:
# Evaluate
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=target_classes))

if len(target_classes) == 2:
    auc = roc_auc_score(y_test, y_proba[:, 1])
    print(f'ROC-AUC: {auc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred),
    display_labels=target_classes
)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=13)

# Feature importances
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
top_n = min(15, len(importances))
importances.tail(top_n).plot(kind='barh', ax=axes[1], color='#2563eb')
axes[1].set_title(f'Top {top_n} Feature Importances', fontsize=13)
axes[1].set_xlabel('Importance')

plt.tight_layout()
# plt.savefig('./model_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Evaluation chart saved')

## 5. Save Model for Deployment

In [ ]:
os.makedirs('model', exist_ok=True)

# Save model
joblib.dump(rf, 'model/bus_delay_rf.joblib')
print('Model saved: model/bus_delay_rf.joblib')

# Save metadata (feature names, target classes, encoders)
model_metadata = {
    'feature_names': feature_cols,
    'target_classes': target_classes,
    'categorical_columns': categorical_cols,
    'n_estimators': rf.n_estimators,
    'max_depth': rf.max_depth,
    'dataset': 'Bus-Breakdown-and-Delays-NYC (OpenML ID 43484)'
}

with open('model/model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2)
print('Metadata saved: model/model_metadata.json')

# Save label encoders
joblib.dump(label_encoders, 'model/label_encoders.joblib')
print('Encoders saved: model/label_encoders.joblib')

print('\nAll artifacts ready for deployment!')
print(f'   Features: {len(feature_cols)}')
print(f'   Classes: {target_classes}')

In [ ]:
# Quick sanity check — simulate a prediction
sample = X_test.iloc[:1].copy()
loaded_model = joblib.load('model/bus_delay_rf.joblib')
pred_class = loaded_model.predict(sample)[0]
pred_proba = loaded_model.predict_proba(sample)[0]

print('=== Sample Prediction ===')
print(f'Predicted class index: {pred_class}')
print(f'Predicted class name: {target_classes[pred_class]}')
print(f'Probabilities: { {target_classes[i]: round(p, 3) for i, p in enumerate(pred_proba)} }')
print('\nModel deployment-ready!')